# Lab 001 · `jax.jit` 的 CPU 执行路径（Jupyter）

这个 Notebook 与 CLI、Marimo 共用 `lab_core.py`。调整控件后，可以交互地查看同一个函数生成的 Jaxpr、StableHLO，以及 shape 变化对 tracing cache 的影响。

In [ ]:
from pathlib import Path
import sys

from IPython.display import Markdown, display
import ipywidgets as widgets

candidates = (Path.cwd(), Path.cwd() / "labs" / "001-jit-cpu")
LAB_DIR = next(path.resolve() for path in candidates if (path / "lab_core.py").is_file())
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from lab_core import cache_experiment
from lab_core import environment_info
from lab_core import jaxpr_text
from lab_core import source_excerpt
from lab_core import source_rows
from lab_core import stablehlo_text

In [ ]:
environment_info()

## 1. 交互观察编译路径

选择阶段并调整输入长度、scale 和 bias。默认先显示 Jaxpr，避免一打开页面就生成较长的 StableHLO。

In [ ]:
stage = widgets.ToggleButtons(
    options=("Jaxpr", "StableHLO", "Run"),
    description="阶段",
)
size = widgets.IntSlider(value=4, min=1, max=16, step=1, description="shape")
scale = widgets.FloatSlider(value=2.0, min=-4.0, max=4.0, step=0.5, description="scale")
bias = widgets.FloatSlider(value=1.0, min=-4.0, max=4.0, step=0.5, description="bias")
output = widgets.Output(layout={"border": "1px solid #ddd", "max_height": "520px", "overflow": "auto"})

def render_stage(_change=None):
    with output:
        output.clear_output(wait=True)
        if stage.value == "Jaxpr":
            print(jaxpr_text(size.value, scale.value, bias.value))
        elif stage.value == "StableHLO":
            print(stablehlo_text(size.value, scale.value, bias.value))
        else:
            for observation in cache_experiment(size.value, scale.value, bias.value):
                print(
                    f"call={observation.call} shape={observation.shape} "
                    f"trace_count={observation.trace_count} result={observation.result}"
                )

for control in (stage, size, scale, bias):
    control.observe(render_stage, names="value")

render_stage()
display(widgets.VBox((stage, widgets.HBox((size, scale, bias)), output)))

## 2. 映射回当前 editable 源码

下拉菜单读取的不是复制到 Notebook 的示例，而是当前 uv 环境实际加载的 JAX symbol。

In [ ]:
rows = source_rows()
source_picker = widgets.Dropdown(
    options=[row["Symbol"] for row in rows],
    value="jax.jit",
    description="Symbol",
    layout={"width": "520px"},
)
source_output = widgets.Output(layout={"border": "1px solid #ddd", "max_height": "520px", "overflow": "auto"})

def render_source(_change=None):
    location, excerpt = source_excerpt(source_picker.value)
    with source_output:
        source_output.clear_output(wait=True)
        display(Markdown(f"`{location}`\n\n```python\n{excerpt}\n```"))

source_picker.observe(render_source, names="value")
render_source()
display(widgets.VBox((source_picker, source_output)))

## 3. 可自动执行的验收

这一格既展示核心结论，也让 `nbconvert --execute` 在结论不成立时直接失败。

In [ ]:
jaxpr = jaxpr_text(4)
stablehlo = stablehlo_text(4)
observations = cache_experiment(4)
trace_counts = [item.trace_count for item in observations]

assert "mul" in jaxpr and "add" in jaxpr
assert "stablehlo.multiply" in stablehlo
assert "stablehlo.add" in stablehlo
assert trace_counts == [1, 1, 2]
assert observations[0].result == [1.0, 3.0, 5.0, 7.0]

{
    "Jaxpr primitives": "mul + add",
    "StableHLO ops": "stablehlo.multiply + stablehlo.add",
    "trace counts": trace_counts,
    "result": observations[0].result,
}

## 4. 连接真实 Hack

本 Lab 的 `patches/trace-pjit.patch` 会在真正的 `_trace_for_jit` 入口加入环境变量控制的观测点。它属于 Python-level Hack，editable 安装会立即读取修改，因此不需要手动编译。

```bash
git -C upstream/jax apply ../../labs/001-jit-cpu/patches/trace-pjit.patch
JAX_SOURCE_ANALYSIS_TRACE_JIT=counted_affine \
  uv run python labs/001-jit-cpu/probe.py --stage run
git -C upstream/jax apply -R ../../labs/001-jit-cpu/patches/trace-pjit.patch
```

修改 jaxlib/XLA/C++ 的后续 Lab 才进入手动编译阶段。